In [1]:
%%capture --no-stderr
!pip install langchain-opentutorial

In [2]:
# Install required packages
from langchain_opentutorial import package

package.install(
    [
        "langchain",
        "langchain_community",
        "chardet"
    ],
    verbose=False,
    upgrade=False,
)

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.errors import (
    GraphRecursionError, 
    InvalidUpdateError, 
    GraphInterrupt, 
    NodeInterrupt, 
    GraphDelegate,
    EmptyInputError,
    TaskNotFound,
    CheckpointNotLatest,
    MultipleSubgraphsError
)



In [25]:
# GraphRecursionError

from langgraph.graph import StateGraph
from typing import TypedDict

# 그래프 상태 정의
class GraphState(TypedDict):
    value: int

# 노드 함수 정의
def increment_node(state: GraphState) -> GraphState:
    state['value'] += 1
    return state

def loop_node(state: GraphState) -> GraphState:
    # 무한 루프를 유발하는 노드
    return state

# 그래프 빌드
builder = StateGraph(GraphState)
builder.add_node("increment", increment_node)
builder.add_node("loop", loop_node)

# 노드 연결
builder.add_edge("increment", "loop")
builder.add_edge("loop", "loop")  # 자기 자신을 가리키는 엣지로 무한 루프 생성

# 그래프 시작점과 종료점 설정
builder.set_entry_point("increment")
builder.set_finish_point("loop")

# 그래프 컴파일
graph = builder.compile()

# 그래프 실행 및 예외 처리
try:
    initial_state = {"value": 0}
    graph.invoke(initial_state, {"recursion_limit": 10})
except GraphRecursionError as e:
    print(f"오류 발생: {e}")


오류 발생: Recursion limit of 10 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT


In [22]:
#InvalidUpdateError

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.errors import InvalidUpdateError

# 그래프 상태 정의
class GraphState(TypedDict):
    foo: int

# 노드 함수 정의
def node_1(state: GraphState):
    print("---Node 1---")
    return {"foo": state["foo"] + 1}

def node_2(state: GraphState):
    print("---Node 2---")
    return {"foo": state["foo"] + 1}

def node_3(state: GraphState):
    print("---Node 3---")
    return {"foo": state["foo"] + 1}

# 그래프 빌드
builder = StateGraph(GraphState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 노드 연결
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 그래프 컴파일
graph = builder.compile()

# 그래프 실행 및 예외 처리
try:
    graph.invoke({"foo": 1})
except InvalidUpdateError as e:
    print(f"InvalidUpdateError 발생: {e}")


---Node 1---
---Node 3---
---Node 2---
InvalidUpdateError 발생: At key 'foo': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_CONCURRENT_GRAPH_UPDATE


In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import NodeInterrupt, GraphInterrupt
import uuid

# 그래프 상태 정의
class GraphState(TypedDict):
    user_input: str
    processed_output: str

# 사용자 입력을 받는 노드 함수 정의
def input_node(state: GraphState) -> GraphState:
    if not state['user_input']:
        # 사용자 입력이 없을 경우 NodeInterrupt 발생
        raise NodeInterrupt("사용자 입력이 필요합니다.")
    return state

# 입력을 처리하는 노드 함수 정의
def process_node(state: GraphState) -> GraphState:
    state['processed_output'] = state['user_input'].upper()
    return state

# 그래프 빌드
builder = StateGraph(GraphState)
builder.add_node("input_node", input_node)
builder.add_node("process_node", process_node)
builder.add_edge(START, "input_node")
builder.add_edge("input_node", "process_node")
builder.add_edge("process_node", END)

# 체크포인터 설정
checkpointer = MemorySaver()

# 그래프 컴파일
graph = builder.compile(checkpointer=checkpointer)

# 스레드 설정
thread_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# 그래프 실행 및 예외 처리
initial_state = {"user_input": "", "processed_output": ""}
try:
    graph.invoke(initial_state, config=thread_config)
except GraphInterrupt as e:
    print(f"GraphInterrupt 발생: {e}")
    # 사용자 입력을 받아 그래프 상태 업데이트
    user_input = input("사용자 입력을 제공하세요: ")
    graph.update_state(config=thread_config, values={"user_input": user_input})
    # 그래프 실행 재개
    graph.invoke(None, config=thread_config)

# 최종 결과 출력
final_state = graph.get_state(thread_config).values
print(f"최종 상태: {final_state}")


최종 상태: {'user_input': '', 'processed_output': ''}


In [6]:
from typing import Any, Dict, Optional
from langgraph.pregel.loop import PregelLoop
from langgraph.errors import GraphDelegate
from langgraph.constants import CONF, CONFIG_KEY_DELEGATE

class GraphDelegateHandler:
    def __init__(self, nodes: Dict, channels: Dict):
        """
        GraphDelegate 처리를 위한 핸들러 초기화
        
        Args:
            nodes: 그래프 노드 정의
            channels: 채널 정의
        """
        self.nodes = nodes
        self.channels = channels
        self.max_delegation_depth = 10  # 최대 위임 깊이 제한
        
    def execute_graph(self, config: Dict, input_data: Any, delegation_depth: int = 0) -> Any:
        """
        그래프 실행 및 위임 처리
        
        Args:
            config: 그래프 설정
            input_data: 입력 데이터
            delegation_depth: 현재 위임 깊이
            
        Returns:
            그래프 실행 결과
        """
        if delegation_depth >= self.max_delegation_depth:
            raise RuntimeError("Maximum delegation depth exceeded")
            
        try:
            with PregelLoop(
                input=input_data,
                config=config,
                nodes=self.nodes,
                specs=self.channels,
                stream=None,
                store=None,
                checkpointer=None,
                output_keys="output",
                stream_keys="stream"
            ) as loop:
                while True:
                    try:
                        # tick 실행 및 결과 확인
                        needs_more = loop.tick(input_keys="input")
                        if not needs_more:
                            return loop.output
                            
                    except GraphDelegate as e:
                        # 위임 정보 추출
                        delegate_info = e.args[0]
                        delegate_config = delegate_info["config"]
                        delegate_input = delegate_info["input"]
                        
                        # 위임된 작업 재귀적 실행
                        result = self.execute_graph(
                            delegate_config,
                            delegate_input,
                            delegation_depth + 1
                        )
                        
                        # 결과를 현재 루프에 반영
                        loop.input = result
                        
        except Exception as e:
            print(f"Error in graph execution: {e}")
            raise

def example_usage():
    """사용 예시"""
    
    # 예시 노드 정의
    def process_node(inputs: Dict) -> Dict:
        return {"output": inputs["input"] * 2}
    
    nodes = {
        "process": process_node
    }
    
    # 예시 채널 정의
    channels = {
        "input": {"type": "input"},
        "output": {"type": "output"}
    }
    
    # 핸들러 초기화
    handler = GraphDelegateHandler(nodes, channels)
    
    # 실행 설정
    config = {
        CONF: {
            CONFIG_KEY_DELEGATE: True  # 위임 활성화
        },
        "metadata": {}
    }
    
    # 테스트 실행
    try:
        result = handler.execute_graph(config, input_data=5)
        print(f"Final result: {result}")
        
    except Exception as e:
        print(f"Error in example: {e}")

if __name__ == "__main__":
    example_usage()

Error in graph execution: 'PregelLoop' object does not support the context manager protocol
Error in example: 'PregelLoop' object does not support the context manager protocol


In [7]:
from typing import Any, Dict, Optional, Union, Sequence
from langgraph.pregel.loop import PregelLoop
from langgraph.errors import EmptyInputError
from langgraph.constants import CONF, CONFIG_KEY_RESUMING, INPUT

class EmptyInputHandler:
    """EmptyInputError 처리를 위한 예제 클래스"""
    
    def __init__(
        self,
        nodes: Dict,
        channels: Dict,
        output_keys: Union[str, Sequence[str]] = "output",
        default_value: Optional[Any] = None
    ):
        self.nodes = nodes
        self.channels = channels
        self.output_keys = output_keys
        self.default_value = default_value
        
    def run(self, input_data: Any) -> Any:
        """
        입력 처리 실행 및 EmptyInputError 처리
        
        Args:
            input_data: 처리할 입력 데이터
            
        Returns:
            처리된 결과
            
        Raises:
            EmptyInputError: fallback 처리 실패 시
            ValueError: 입력 검증 실패 시
        """
        config = {
            CONF: {
                CONFIG_KEY_RESUMING: False
            },
            "metadata": {}
        }
        
        try:
            with PregelLoop(
                input=input_data,
                config=config,
                nodes=self.nodes,
                specs=self.channels,
                stream=None,
                store=None,
                checkpointer=None,
                output_keys=self.output_keys
            ) as loop:
                try:
                    while loop.tick(input_keys=INPUT):
                        pass
                    return loop.output
                    
                except EmptyInputError as e:
                    print(f"EmptyInputError caught: {e}")
                    
                    # default_value가 있으면 재시도
                    if self.default_value is not None:
                        print(f"Retrying with default value: {self.default_value}")
                        loop.input = self.default_value
                        while loop.tick(input_keys=INPUT):
                            pass
                        return loop.output
                    else:
                        raise EmptyInputError("No default value available for empty input")
                        
        except Exception as e:
            print(f"Error during processing: {type(e).__name__}: {e}")
            raise

def example():
    """사용 예시"""
    
    # 1. 노드 정의
    def process_node(inputs: Dict) -> Dict:
        """입력을 처리하는 예시 노드"""
        if not inputs.get(INPUT):
            raise ValueError("Empty input in process node")
        return {"output": f"Processed: {inputs[INPUT]}"}
    
    nodes = {
        "process": process_node
    }
    
    # 2. 채널 정의 
    channels = {
        INPUT: {"type": "input"},
        "output": {"type": "output"}
    }
    
    # 3. 핸들러 생성
    handler = EmptyInputHandler(
        nodes=nodes,
        channels=channels,
        default_value="default_input"
    )
    
    # 4. 다양한 입력 케이스 테스트
    test_cases = [
        ("valid_input", "정상 입력 테스트"),
        (None, "None 입력 테스트"),
        ("", "빈 문자열 테스트"),
        ({}, "빈 딕셔너리 테스트")
    ]
    
    # 5. 테스트 실행
    for input_data, desc in test_cases:
        print(f"\n=== {desc} ===")
        try:
            result = handler.run(input_data)
            print(f"Result: {result}")
        except EmptyInputError as e:
            print(f"EmptyInputError handled: {e}")
        except Exception as e:
            print(f"Other error: {type(e).__name__}: {e}")

if __name__ == "__main__":
    example()


=== 정상 입력 테스트 ===
Error during processing: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'
Other error: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'

=== None 입력 테스트 ===
Error during processing: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'
Other error: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'

=== 빈 문자열 테스트 ===
Error during processing: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'
Other error: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'

=== 빈 딕셔너리 테스트 ===
Error during processing: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'
Other error: TypeError: PregelLoop.__init__() missing 1 required keyword-only argument: 'stream_keys'


In [9]:
from typing import Any, Dict, Optional, Union, Sequence
from langgraph.pregel.loop import PregelLoop
from langgraph.errors import CheckpointNotLatest
from langgraph.constants import CONF, CONFIG_KEY_CHECKPOINT_ID, CONFIG_KEY_ENSURE_LATEST
from langgraph.checkpoint.base import BaseCheckpointSaver

class CheckpointHandler:
    def __init__(
        self,
        nodes: Dict,
        channels: Dict,
        output_keys: Union[str, Sequence[str]] = "output",
        stream_keys: Union[str, Sequence[str]] = "stream"
    ):
        """
        CheckpointNotLatest 처리를 위한 핸들러
        
        Args:
            nodes: 그래프 노드 정의
            channels: 채널 정의
            output_keys: 출력 키
            stream_keys: 스트림 키
        """
        self.nodes = nodes
        self.channels = channels
        self.output_keys = output_keys
        self.stream_keys = stream_keys
        
    def process_with_checkpoint(
        self,
        input_data: Any,
        config: Dict,
        checkpointer: Optional[BaseCheckpointSaver] = None,
        max_retries: int = 3
    ) -> Any:
        """
        체크포인트를 사용한 처리 및 CheckpointNotLatest 처리
        
        Args:
            input_data: 처리할 입력 데이터
            config: 설정
            checkpointer: 체크포인트 저장소
            max_retries: 최대 재시도 횟수
            
        Returns:
            처리 결과
        """
        retry_count = 0
        
        while True:
            try:
                with PregelLoop(
                    input=input_data,
                    config=config,
                    nodes=self.nodes,
                    specs=self.channels,
                    store=None,
                    stream=None,  # 필수 인자 추가
                    checkpointer=checkpointer,
                    output_keys=self.output_keys,
                    stream_keys=self.stream_keys  # 필수 인자 추가
                ) as loop:
                    # 그래프 실행
                    while loop.tick(input_keys="input"):
                        pass
                    return loop.output
                    
            except CheckpointNotLatest:
                retry_count += 1
                print(f"CheckpointNotLatest detected. Retry attempt {retry_count}")
                
                if retry_count >= max_retries:
                    raise RuntimeError(f"Maximum retry attempts ({max_retries}) exceeded")
                
                # 최신 체크포인트 조회
                if checkpointer:
                    latest_checkpoint = checkpointer.get_tuple(
                        {
                            **config,
                            CONF: {
                                **config.get(CONF, {}),
                                CONFIG_KEY_CHECKPOINT_ID: None
                            }
                        }
                    )
                    
                    if latest_checkpoint:
                        # 설정 업데이트
                        config = {
                            **config,
                            CONF: {
                                **config.get(CONF, {}),
                                CONFIG_KEY_CHECKPOINT_ID: latest_checkpoint.checkpoint["id"],
                                CONFIG_KEY_ENSURE_LATEST: True
                            }
                        }
                        continue
                        
                raise  # 체크포인트를 찾을 수 없는 경우 예외 발생

def example_usage():
    """사용 예시"""
    
    # 예시 노드 정의
    def process_node(inputs: Dict) -> Dict:
        return {"output": inputs["input"]}
    
    nodes = {
        "process": process_node
    }
    
    # 예시 채널 정의
    channels = {
        "input": {"type": "input"},
        "output": {"type": "output"}
    }
    
    # 핸들러 초기화
    handler = CheckpointHandler(
        nodes=nodes,
        channels=channels,
        output_keys="output",
        stream_keys="stream"
    )
    
    # 실행 설정
    config = {
        CONF: {
            CONFIG_KEY_CHECKPOINT_ID: "some_checkpoint_id",
            CONFIG_KEY_ENSURE_LATEST: True
        }
    }
    
    # 테스트 실행
    try:
        result = handler.process_with_checkpoint(
            input_data={"input": "test_data"},
            config=config,
            checkpointer=None  # 실제 환경에서는 체크포인트 저장소 제공
        )
        print(f"Final result: {result}")
        
    except Exception as e:
        print(f"Error in example: {e}")

if __name__ == "__main__":
    example_usage()

Error in example: 'PregelLoop' object does not support the context manager protocol


In [15]:
from typing import Any, Dict, Optional, Union, Sequence, Set, TypeVar, Type
from langgraph.pregel.loop import SyncPregelLoop
from langgraph.errors import MultipleSubgraphsError
from langgraph.constants import (
    CONF,
    CONFIG_KEY_CHECKPOINT_NS,
    CONFIG_KEY_TASK_ID
)
from langgraph.channels.base import BaseChannel
from langgraph.graph.state import StateChannel

class SubgraphHandler:
    """서브그래프 실행 및 MultipleSubgraphsError 처리를 위한 핸들러"""
    
    def __init__(
        self,
        nodes: Dict,
        channels: Dict[str, BaseChannel],
        output_keys: Union[str, Sequence[str]] = "output",
        stream_keys: Union[str, Sequence[str]] = "stream"
    ):
        self.nodes = nodes
        self.channels = channels
        self.output_keys = output_keys
        self.stream_keys = stream_keys
        self.seen_namespaces: Set[str] = set()
        
    def validate_namespace(self, namespace: str) -> None:
        """네임스페이스 중복 검사"""
        if namespace in self.seen_namespaces:
            raise MultipleSubgraphsError(
                f"Multiple subgraphs called with namespace '{namespace}'\n\n"
                "Troubleshooting:\n"
                "1. 하나의 노드에서는 하나의 서브그래프만 호출해야 합니다\n"
                "2. 서브그래프를 순차적으로 실행하려면 별도의 노드로 분리하세요\n"
                "3. 각 서브그래프는 고유한 네임스페이스를 가져야 합니다"
            )
        self.seen_namespaces.add(namespace)
        
    def execute_subgraph(
        self,
        input_data: Any,
        namespace: str,
        parent_config: Dict,
        task_id: Optional[str] = None
    ) -> Any:
        """서브그래프 실행"""
        try:
            # 네임스페이스 검증
            self.validate_namespace(namespace)
            
            # 서브그래프 설정 생성
            config = {
                **parent_config,
                CONF: {
                    **parent_config.get(CONF, {}),
                    CONFIG_KEY_CHECKPOINT_NS: namespace,
                    CONFIG_KEY_TASK_ID: task_id
                }
            }
            
            # SyncPregelLoop으로 서브그래프 실행
            with SyncPregelLoop(
                input=input_data,
                config=config,
                nodes=self.nodes,
                specs=self.channels,
                store=None,
                stream=None,
                checkpointer=None,
                output_keys=self.output_keys,
                stream_keys=self.stream_keys,
                check_subgraphs=True
            ) as loop:
                while loop.tick(input_keys="input"):
                    pass
                return loop.output
                
        except MultipleSubgraphsError as e:
            print(f"MultipleSubgraphsError caught: {e}")
            raise
            
        finally:
            # 실행 완료 후 네임스페이스 정리
            if namespace in self.seen_namespaces:
                self.seen_namespaces.remove(namespace)

def example():
    """사용 예시"""
    
    # 1. 노드 정의
    def process_node(inputs: Dict) -> Dict:
        return {"output": f"Processed: {inputs['input']}"}
    
    nodes = {
        "process": process_node
    }
    
    # 2. 채널 정의 - StateChannel 사용
    channels = {
        "input": StateChannel(),
        "output": StateChannel()
    }
    
    # 3. 핸들러 생성
    handler = SubgraphHandler(
        nodes=nodes,
        channels=channels,
        output_keys="output",
        stream_keys="stream"
    )
    
    # 4. 테스트 시나리오
    def run_tests():
        base_config = {
            CONF: {},
            "metadata": {"graph_type": "example"},
            "recursion_limit": 100
        }
        
        try:
            # 정상 케이스: 서로 다른 네임스페이스
            print("\n=== Test 1: Different namespaces ===")
            result1 = handler.execute_subgraph(
                input_data={"input": "data1"},
                namespace="graph1",
                parent_config=base_config
            )
            print(f"First graph result: {result1}")
            
            result2 = handler.execute_subgraph(
                input_data={"input": "data2"},
                namespace="graph2",
                parent_config=base_config
            )
            print(f"Second graph result: {result2}")
            
            # 에러 케이스: 중복 네임스페이스
            print("\n=== Test 2: Duplicate namespace ===")
            handler.execute_subgraph(
                input_data={"input": "data3"},
                namespace="graph1",  # 중복 네임스페이스
                parent_config=base_config
            )
            
        except MultipleSubgraphsError as e:
            print(f"Expected error: {e}")
            
        except Exception as e:
            print(f"Unexpected error: {type(e).__name__}: {e}")
            raise
            
        print("\n=== Test 3: Namespace cleanup ===")
        print(f"Remaining namespaces: {handler.seen_namespaces}")
        
    run_tests()

if __name__ == "__main__":
    example()

ImportError: cannot import name 'StateChannel' from 'langgraph.graph.state' (/Users/pyoungwonseo/Library/Caches/pypoetry/virtualenvs/langchain-opentutorial-99wpaVyw-py3.11/lib/python3.11/site-packages/langgraph/graph/state.py)